# 11. Counting, and "have I seen this before?"

After this notebook you'll be able to use a dictionary to:

- **count** how many times each thing appears,
- use those counts to answer questions (most common? appears only once? same letters?),
- **remember what you've already seen**, and where, while you walk through a list once.

These two patterns solve a big share of LeetCode Easy problems, including **Two Sum**, the most famous one. Take your time here. Run every cell, and predict before you run.

## Part 1: counting

### Tally marks

Your class votes for the trip: hills, beach or zoo. The teacher reads the slips one by one and keeps a tally on the board:

```
hills  |||
beach  ||
zoo    |
```

For each slip, the teacher does the same thing:

1. Find that place on the board. If it isn't there yet, write it with **0** marks.
2. Add **one** mark.

A dictionary does exactly this. The **key** is the thing you're counting (the place), and the **value** is how many you've seen so far (the marks).

### First try: it breaks

Here's the obvious way to add one mark. Run it.

In [ ]:
votes = ["hills", "beach", "hills"]
counts = {}
for place in votes:
    counts[place] += 1
print(counts)

```
KeyError: 'hills'
```

`counts[place] += 1` means `counts[place] = counts[place] + 1`. The right-hand side **reads** `counts["hills"]`, but the board is empty. There's no `"hills"` yet, so: KeyError.

The teacher's step 1 ("if it isn't there, write it with 0") is missing.

### Second try: with `if`

Add step 1 with an `if`:

In [ ]:
votes = ["hills", "beach", "hills", "zoo", "beach", "hills"]
counts = {}
for place in votes:
    if place in counts:
        counts[place] += 1     # seen before: one more mark
    else:
        counts[place] = 1      # first time: start at 1
print(counts)

```
{'hills': 3, 'beach': 2, 'zoo': 1}
```

It works. This version is a fine answer, and it's the easiest to understand.

### The short version: `get(key, 0)`

Programmers usually write those four lines as one:

```python
counts[place] = counts.get(place, 0) + 1
```

Read it from the right:

1. `counts.get(place, 0)`: how many have I counted so far? (`0` if this is the first one)
2. `+ 1`: one more.
3. `counts[place] = ...`: store the new number back.

It does exactly what the `if`/`else` did. Run it, with a `print` inside the loop so you can watch the dictionary grow:

In [ ]:
votes = ["hills", "beach", "hills", "zoo", "beach", "hills"]
counts = {}
for place in votes:
    counts[place] = counts.get(place, 0) + 1
    print(place, "->", counts)
print("final:", counts)

```
hills -> {'hills': 1}
beach -> {'hills': 1, 'beach': 1}
hills -> {'hills': 2, 'beach': 1}
zoo -> {'hills': 2, 'beach': 1, 'zoo': 1}
beach -> {'hills': 2, 'beach': 2, 'zoo': 1}
hills -> {'hills': 3, 'beach': 2, 'zoo': 1}
final: {'hills': 3, 'beach': 2, 'zoo': 1}
```

The same thing as a table:

| Step | `place` | `counts.get(place, 0)` | stored | `counts` after this step |
| --- | --- | --- | --- | --- |
| 1 | `"hills"` | `0` | `1` | `{'hills': 1}` |
| 2 | `"beach"` | `0` | `1` | `{'hills': 1, 'beach': 1}` |
| 3 | `"hills"` | `1` | `2` | `{'hills': 2, 'beach': 1}` |
| 4 | `"zoo"` | `0` | `1` | `{'hills': 2, 'beach': 1, 'zoo': 1}` |
| 5 | `"beach"` | `1` | `2` | `{'hills': 2, 'beach': 2, 'zoo': 1}` |
| 6 | `"hills"` | `2` | `3` | `{'hills': 3, 'beach': 2, 'zoo': 1}` |

Look at the `counts.get(place, 0)` column: it's `0` the first time each place appears, and the old count after that.

### Counting letters

A string works the same way, because a `for` loop over a string gives you one letter at a time. **Predict** the final dictionary for `"banana"` before you run it. Write it on paper if that helps.

In [ ]:
counts = {}
for ch in "banana":
    counts[ch] = counts.get(ch, 0) + 1
print(counts)
print(counts["a"])
print(counts.get("z", 0))

```
{'b': 1, 'a': 3, 'n': 2}
3
0
```

- The keys are in the order each letter was **first** seen: `b`, then `a`, then `n`.
- `counts.get("z", 0)` is `0`: no `z` in banana, and no KeyError.

**Try it:** count the letters in your own name.

## Using the counts

Counting is usually step one. Step two is asking a question about the counts. Here are the three questions that come up most.

### Question 1: which one appears most?

Loop over the pairs with `.items()`, and keep the best one so far (like the most expensive item in the last notebook):

In [ ]:
counts = {"hills": 3, "beach": 2, "zoo": 1}

best_place = None
best_count = 0
for place, n in counts.items():
    if n > best_count:
        best_place = place
        best_count = n
print(best_place, best_count)

```
hills 3
```

| `place` | `n` | `n > best_count`? | `best_place`, `best_count` after |
| --- | --- | --- | --- |
| (start) | | | `None`, `0` |
| `"hills"` | `3` | yes (3 > 0) | `"hills"`, `3` |
| `"beach"` | `2` | no (2 > 3 is false) | `"hills"`, `3` |
| `"zoo"` | `1` | no | `"hills"`, `3` |

### Question 2: which ones appear only once?

This needs **two loops**: the first loop counts everything, the second loop asks questions. You can't know whether a letter is unique until you've seen the **whole** word.

Find the first letter in `"swiss"` that appears only once:

In [ ]:
word = "swiss"

counts = {}
for ch in word:                 # loop 1: count
    counts[ch] = counts.get(ch, 0) + 1
print(counts)

for ch in word:                 # loop 2: ask
    print(ch, "appears", counts[ch], "time(s)")
    if counts[ch] == 1:
        print("first unique letter:", ch)
        break

```
{'s': 3, 'w': 1, 'i': 1}
s appears 3 time(s)
w appears 1 time(s)
first unique letter: w
```

Why not do it in one loop? At the first `s`, its count so far is `1`, so it **looks** unique. Only after the whole word do you know there are three. Count first, then ask.

The second loop goes over the **word**, not the dictionary, because the question is about the order in the word.

### Question 3: same letters, same number of times?

Two words are **anagrams** when they use exactly the same letters, the same number of times: `"listen"` and `"silent"`.

Two dictionaries are equal (`==`) when they have the same pairs. The order of the pairs doesn't matter. Predict:

In [ ]:
print({"a": 1, "b": 2} == {"b": 2, "a": 1})
print({"a": 1, "b": 2} == {"a": 2, "b": 1})

```
True
False
```

So: count both words, then compare the two dictionaries.

In [ ]:
def count_letters(word):
    counts = {}
    for ch in word:
        counts[ch] = counts.get(ch, 0) + 1
    return counts

print(count_letters("listen"))
print(count_letters("silent"))
print(count_letters("listen") == count_letters("silent"))
print(count_letters("cat") == count_letters("cart"))

```
{'l': 1, 'i': 1, 's': 1, 't': 1, 'e': 1, 'n': 1}
{'s': 1, 'i': 1, 'l': 1, 'e': 1, 'n': 1, 't': 1}
True
False
```

The two dictionaries are printed in different orders, but they hold the same pairs, so `==` says `True`.

Putting the counting loop in its own function means you can use it again and again. You'll do that in the tasks.

## Part 2: have I seen this before, and where?

### A visitor book

At the gate of a building, a guard writes down every visitor and the time they came in. When someone walks up, the guard first **checks the book**: "Have you been here already today? Yes, at 9:15." Then, if they're new, the guard **writes them in**.

The order matters: **check first, then write**. If the guard wrote the name first and then checked, the book would always say "yes, you're here", because the guard had written it a second ago.

In code, the book is a dictionary called `seen`:

- the **key** is a value from the list (the visitor),
- the **value** is the **index** where you saw it (the time).

You walk through the list **once**, and at every step you can ask "have I met this number before? Where?"

In [ ]:
nums = [5, 3, 8, 3, 1]
seen = {}
for i, n in enumerate(nums):
    print("step", i, ": n =", n, ", seen =", seen)
    if n in seen:
        print(n, "was first seen at index", seen[n], "and again at index", i)
        break
    seen[n] = i

```
step 0 : n = 5 , seen = {}
step 1 : n = 3 , seen = {5: 0}
step 2 : n = 8 , seen = {5: 0, 3: 1}
step 3 : n = 3 , seen = {5: 0, 3: 1, 8: 2}
3 was first seen at index 1 and again at index 3
```

| `i` | `n` | `n in seen`? | what happens | `seen` afterwards |
| --- | --- | --- | --- | --- |
| 0 | 5 | no | remember `5` at index 0 | `{5: 0}` |
| 1 | 3 | no | remember `3` at index 1 | `{5: 0, 3: 1}` |
| 2 | 8 | no | remember `8` at index 2 | `{5: 0, 3: 1, 8: 2}` |
| 3 | 3 | **yes**, at 1 | print and stop | |

### What goes wrong if you write first

Swap the two steps: remember first, then check. Predict what it prints, then run it.

In [ ]:
nums = [5, 3, 8, 3, 1]
seen = {}
for i, n in enumerate(nums):
    seen[n] = i               # remember first (wrong order)
    if n in seen:
        print("repeat found:", n, "at index", seen[n], "and", i)
        break

```
repeat found: 5 at index 0 and 0
```

Wrong. `5` isn't repeated at all. The loop wrote `5` into `seen`, then immediately found it there. **Check first, then remember.**

### Two Sum, step by step

This is LeetCode problem 1. You get a list `nums` and a number `target`. Find the **two positions** whose numbers add up to `target`. There's always exactly one answer, and you can't use the same position twice.

```
nums = [2, 7, 11, 15], target = 9   ->  [0, 1]    because 2 + 7 = 9
```

**The slow way** checks every pair with a loop inside a loop. It works, but on a list of 10,000 numbers that's about 50 million pairs.

**The fast way** walks through the list once, with a `seen` dictionary. The question at each step changes slightly. Not "have I seen **this** number?" but:

> "Have I seen the number that would **complete** my answer?"

If the current number is `n`, the number you need is `target - n`. Call it `need`.

In [ ]:
def two_sum(nums, target):
    seen = {}                        # number -> its index
    for i, n in enumerate(nums):
        need = target - n
        print("i =", i, ", n =", n, ", need =", need, ", seen =", seen)
        if need in seen:             # check first...
            return [seen[need], i]
        seen[n] = i                  # ...then remember

print(two_sum([2, 7, 11, 15], 9))

```
i = 0 , n = 2 , need = 7 , seen = {}
i = 1 , n = 7 , need = 2 , seen = {2: 0}
[0, 1]
```

| `i` | `n` | `need = 9 - n` | `need in seen`? | what happens | `seen` afterwards |
| --- | --- | --- | --- | --- | --- |
| 0 | 2 | 7 | no | remember 2 at 0 | `{2: 0}` |
| 1 | 7 | 2 | **yes**, at 0 | return `[0, 1]` | |

At step 1, the `7` doesn't need to search the list for a `2`. The dictionary already knows: "2 is at index 0".

Notice the answer is `[seen[need], i]`: the **older** position first (where `need` was), then the current one.

A longer one. Predict where it stops before you run it.

In [ ]:
print(two_sum([4, 1, 9, 6, 3], 9))

```
i = 0 , n = 4 , need = 5 , seen = {}
i = 1 , n = 1 , need = 8 , seen = {4: 0}
i = 2 , n = 9 , need = 0 , seen = {4: 0, 1: 1}
i = 3 , n = 6 , need = 3 , seen = {4: 0, 1: 1, 9: 2}
i = 4 , n = 3 , need = 6 , seen = {4: 0, 1: 1, 9: 2, 6: 3}
[3, 4]
```

`6 + 3 = 9`. The `3` at index 4 needed a `6`, and `seen` said `6` was at index 3.

### Why the order matters in Two Sum

Here's the trap. `nums = [3, 2, 4]`, `target = 6`. The right answer is `[1, 2]` (2 + 4).

If you **remember first, then check**, the `3` at index 0 needs a `3`... and finds itself. Run the broken version:

In [ ]:
def two_sum_wrong(nums, target):
    seen = {}
    for i, n in enumerate(nums):
        seen[n] = i                  # remember first (wrong order)
        need = target - n
        if need in seen:
            return [seen[need], i]

print(two_sum_wrong([3, 2, 4], 6))
print(two_sum([3, 2, 4], 6))

```
[0, 0]
i = 0 , n = 3 , need = 3 , seen = {}
i = 1 , n = 2 , need = 4 , seen = {3: 0}
i = 2 , n = 4 , need = 2 , seen = {3: 0, 2: 1}
[1, 2]
```

The wrong version answered `[0, 0]`: it used the same `3` twice. The right version only looks at numbers **before** the current one, so a number can't pair with itself.

## A shortcut: `Counter`

Counting is so common that Python has a ready-made counter. It lives in a toolbox called `collections`, so you **import** it first (bring it into your code):

In [ ]:
from collections import Counter

counts = Counter("banana")
print(counts)
print(counts["a"])
print(counts["z"])
print(Counter("listen") == Counter("silent"))

```
Counter({'a': 3, 'n': 2, 'b': 1})
3
0
True
```

A `Counter` works like the dictionary you built by hand, with one difference: a missing key gives `0`, not a KeyError.

LeetCode allows it. But write the counting loop yourself in the tasks below until it feels easy. Notebook 21 covers `Counter` properly.

## Common mistakes

**`counts[x] += 1` on an empty dictionary.** KeyError, because the key isn't there yet. Use `counts[x] = counts.get(x, 0) + 1`.

**Asking a question before counting is finished.** "Does this letter appear once?" can only be answered after the whole counting loop. Count in one loop, ask in a second.

**A `return` inside the counting loop.** Then the function stops after the first item. Run this and look at the answer:

In [ ]:
def count_letters_wrong(word):
    counts = {}
    for ch in word:
        counts[ch] = counts.get(ch, 0) + 1
        return counts                 # inside the loop: too early

print(count_letters_wrong("banana"))

```
{'b': 1}
```

The `return` should be lined up with `for`, not inside it.

**Remembering before checking** in the "seen" pattern. The number finds itself. Check first, then remember.

---

# ✏️ Your turn

The first tasks practise the pattern; the last four are real LeetCode Easy problems. Write your code, run the cell, then run the check cell under it.

If a task feels hard, scroll up to the matching example and trace it again with a small input on paper.

### Task 1: count the words

Write `count_words(words)`. `words` is a list of strings. Return a dictionary of how many times each word appears.

```
count_words(["tea", "coffee", "tea"])   ->  {"tea": 2, "coffee": 1}
count_words([])                         ->  {}
```

In [ ]:
def count_words(words):
    # your code here
    pass

In [ ]:
from checks import check
check(count_words(["tea", "coffee", "tea"]), {"tea": 2, "coffee": 1})
check(count_words(["cat", "cat", "cat"]), {"cat": 3})
check(count_words([]), {})

### Task 2: letters that repeat

Write `repeated_letters(word)`. Return a list of the letters that appear **more than once**, in the order they first appear in the word.

```
repeated_letters("banana")   ->  ["a", "n"]
repeated_letters("hello")    ->  ["l"]
repeated_letters("cat")      ->  []
```

Hint: count first. Then loop over the counts with `.items()`.

In [ ]:
def repeated_letters(word):
    # your code here
    pass

In [ ]:
check(repeated_letters("banana"), ["a", "n"])
check(repeated_letters("hello"), ["l"])
check(repeated_letters("mississippi"), ["i", "s", "p"])
check(repeated_letters("cat"), [])

### Task 3: the winning vote

Write `winner(votes)`. Return the place with the most votes. There's always at least one vote, and no ties.

```
winner(["hills", "beach", "hills"])   ->  "hills"
winner(["zoo"])                       ->  "zoo"
```

In [ ]:
def winner(votes):
    # your code here
    pass

In [ ]:
check(winner(["hills", "beach", "hills"]), "hills")
check(winner(["beach", "zoo", "beach", "hills", "beach"]), "beach")
check(winner(["zoo"]), "zoo")

### Task 4 (LeetCode 169): Majority Element

You get a list `nums`. One number appears **more than half** the time. Return that number.

```
[3, 2, 3]              ->  3
[2, 2, 1, 1, 1, 2, 2]  ->  2
```

Hint: count, then find the number whose count is bigger than `len(nums) // 2`.

In [ ]:
class Solution:
    def majorityElement(self, nums):
        # your code here
        pass

In [ ]:
check(Solution().majorityElement([3, 2, 3]), 3)
check(Solution().majorityElement([2, 2, 1, 1, 1, 2, 2]), 2)
check(Solution().majorityElement([7]), 7)

### Task 5 (LeetCode 387): First Unique Character in a String

You get a string `s`. Return the **index** of the first letter that appears only once. If there isn't one, return `-1`.

```
"leetcode"      ->  0     ("l" appears once)
"loveleetcode"  ->  2     ("v")
"aabb"          ->  -1
```

Hint: two loops, like the `"swiss"` example. The second loop needs the index, so use `enumerate` or `range(len(s))`.

In [ ]:
class Solution:
    def firstUniqChar(self, s):
        # your code here
        pass

In [ ]:
check(Solution().firstUniqChar("leetcode"), 0)
check(Solution().firstUniqChar("loveleetcode"), 2)
check(Solution().firstUniqChar("aabb"), -1)
check(Solution().firstUniqChar("z"), 0)

### Task 6 (LeetCode 242): Valid Anagram

You get two strings `s` and `t`. Return `True` if `t` uses exactly the same letters as `s`, the same number of times. Otherwise `False`.

```
s = "anagram", t = "nagaram"   ->  True
s = "rat",     t = "car"       ->  False
```

Hint: a helper function that counts letters, used twice.

In [ ]:
class Solution:
    def isAnagram(self, s, t):
        # your code here
        pass

In [ ]:
check(Solution().isAnagram("anagram", "nagaram"), True)
check(Solution().isAnagram("rat", "car"), False)
check(Solution().isAnagram("aacc", "ccac"), False)
check(Solution().isAnagram("a", "a"), True)

### Task 7 (LeetCode 383): Ransom Note

You want to write a note by cutting letters out of a magazine. Each letter in the magazine can be used **once**. Return `True` if you can make `ransomNote` from the letters in `magazine`.

```
ransomNote = "a",  magazine = "b"     ->  False
ransomNote = "aa", magazine = "ab"    ->  False   (only one "a")
ransomNote = "aa", magazine = "aab"   ->  True
```

Hint: count the magazine's letters. Then loop over the note: for each letter, if the count is `0` (use `get`), return `False`; otherwise take one away.

In [ ]:
class Solution:
    def canConstruct(self, ransomNote, magazine):
        # your code here
        pass

In [ ]:
check(Solution().canConstruct("a", "b"), False)
check(Solution().canConstruct("aa", "ab"), False)
check(Solution().canConstruct("aa", "aab"), True)
check(Solution().canConstruct("", "abc"), True)

### Task 8 (LeetCode 1): Two Sum

Now write it yourself, without scrolling up. You get `nums` and `target`. Return the two indexes (older one first) whose numbers add up to `target`. There's always exactly one answer.

```
nums = [2, 7, 11, 15], target = 9   ->  [0, 1]
nums = [3, 2, 4],      target = 6   ->  [1, 2]
nums = [3, 3],         target = 6   ->  [0, 1]
```

If you get stuck, say the plan out loud: "for each number, what do I need? Have I seen it? If yes, done. If no, remember this number."

In [ ]:
class Solution:
    def twoSum(self, nums, target):
        # your code here
        pass

In [ ]:
check(Solution().twoSum([2, 7, 11, 15], 9), [0, 1])
check(Solution().twoSum([3, 2, 4], 6), [1, 2])
check(Solution().twoSum([3, 3], 6), [0, 1])
check(Solution().twoSum([1, 5, 8, 2], 10), [2, 3])

---

# ✅ Quiz: check yourself

Put your letter between the quotes, then run the cell and the check cell under it.

**Question 1.** `counts = {"a": 2}`. After `counts["b"] = counts.get("b", 0) + 1`, what is `counts`?

- A) A KeyError
- B) `{"a": 2, "b": 1}`
- C) `{"a": 2, "b": 0}`

In [ ]:
answer_1 = "?"

In [ ]:
from checks import check_answer
check_answer("11-q1", answer_1)

**Question 2.** To find the first letter that appears only once, why do you need two loops?

- A) Because you only know a letter is unique after counting the whole word
- B) Because one loop can't use a dictionary

In [ ]:
answer_2 = "?"

In [ ]:
check_answer("11-q2", answer_2)

**Question 3.** In Two Sum, with `target = 10` and the current number `n = 4`, what do you look up in `seen`?

- A) `4`
- B) `10`
- C) `6`

In [ ]:
answer_3 = "?"

In [ ]:
check_answer("11-q3", answer_3)

**Question 4.** In the "seen before" pattern, what is the right order inside the loop?

- A) Remember the number, then check
- B) Check, then remember the number

In [ ]:
answer_4 = "?"

In [ ]:
check_answer("11-q4", answer_4)

---

## Done

You can now count anything with a dictionary, use the counts to find the most common thing, the unique ones and anagrams, and solve Two Sum in one pass with a `seen` dictionary.

These patterns come back again and again. When a problem says "how many times", "appears once", "same letters" or "find a pair", think of this notebook.

Stuck on a task? The answers are in `solutions/11-counting-and-seen-before.ipynb`. Try for real first.

**Next:** sets (a list with no repeats and a fast "is it in here?") and tuples (values that stay together) (`12-sets-and-tuples`).